# FirstLight_CR GPU benchmark (Kaggle)

Measures IL activation memory and throughput of the V4 policy on a Kaggle GPU, to check the RTX 4060/4070 plan in `docs/training_rtx4060_4070.zh-CN.md`.
Settings: **Accelerator = GPU (T4 or P100)**, **Internet = On**. Run all cells (~15-25 min). Results: `/kaggle/working/bench.jsonl`.

In [ ]:
!nvidia-smi
!git clone -q --depth 1 -b perf/consumer-gpu-training https://github.com/xiaoyuzhoux120-droid/FirstLight_CR.git /kaggle/working/FirstLight_CR
!pip install -q orjson safetensors zstandard sc-compression pyarrow
import torch; print(torch.__version__, torch.cuda.get_device_name(0), 'bf16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
%cd /kaggle/working/FirstLight_CR
# Pick the autocast dtype: T4/P100 have no native bf16, so FP16 (+GradScaler) is the closer proxy for speed; memory per frame is the same for both.
import torch
DTYPE = 'bfloat16' if torch.cuda.is_bf16_supported(including_emulation=False) else 'float16'
print('autocast', DTYPE)

In [ ]:
# 1) How many owner sequences fit in one forward at T=32 (memory per frame)
!python tools/benchmark_training.py --device cuda --autocast $DTYPE --time-steps 32 --sweep 8,16,24,32,48 --iterations 3 --json /kaggle/working/bench.jsonl

In [ ]:
# 2) Same effective batch (48 sequences), different micro-batch counts
for m in (1, 2, 3, 6):
    !python tools/benchmark_training.py --device cuda --autocast $DTYPE --time-steps 32 --batch 48 --micro-batches {m} --iterations 3 --json /kaggle/working/bench.jsonl

In [ ]:
# 3) FP32 reference and the other dtype for comparison
!python tools/benchmark_training.py --device cuda --autocast none --time-steps 32 --sweep 8,16 --iterations 3 --json /kaggle/working/bench.jsonl
!python tools/benchmark_training.py --device cuda --autocast bfloat16 --time-steps 32 --sweep 16 --iterations 3 --json /kaggle/working/bench.jsonl

In [ ]:
# 4) New unit tests on the GPU machine (CPU-side correctness, ~1 min)
!python -m pytest native_runner/tests/test_training_v4_microbatch.py -q -W ignore -p no:cacheprovider

In [ ]:
import json, pandas as pd
rows = [json.loads(l) for l in open('/kaggle/working/bench.jsonl')]
df = pd.DataFrame(rows)
if 'cuda_peak_reserved_mb' in df:
    df['frames_per_pass'] = df['batch'] // df['micro_batches'] * df['time_steps']
    df['mb_per_frame'] = (df['cuda_peak_allocated_mb'] - 250) / df['frames_per_pass']
print(df.to_string())